In [1]:
# Install libraries
%pip install transformers torch pandas scikit-learn -q

import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

from transformers import (
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments
)

# 1. Load dataset
df = pd.read_csv("IMDB Dataset.csv")

# 2. Define sentiment classes
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

df = df.sample(1000, random_state=42)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["label"]
)

# 3 & 4. BERT Tokenization and Input Format
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_tokens = tokenizer(
    train_df["review"].tolist(),
    padding="max_length", truncation=True, max_length=64
)

test_tokens = tokenizer(
    test_df["review"].tolist(),
    padding="max_length", truncation=True, max_length=64
)

# Create PyTorch Dataset
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, tokens, labels):
        self.tokens = tokens
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.tokens["input_ids"][i]),
            "attention_mask": torch.tensor(self.tokens["attention_mask"][i]),
            "labels": torch.tensor(self.labels[i])
        }

train_data = ReviewDataset(train_tokens, train_df["label"])
test_data = ReviewDataset(test_tokens, test_df["label"])

# 5. Load pretrained BERT
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

# 6. Train BERT using Trainer
def metrics(p):
    pred = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, pred),
        "precision": precision_score(p.label_ids, pred, zero_division=0),
        "recall": recall_score(p.label_ids, pred, zero_division=0),
        "f1": f1_score(p.label_ids, pred, zero_division=0)
    }

args = TrainingArguments(
    output_dir="./bert_customer_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=metrics
)

print("Training Started...")
trainer.train()

# 7 & 8. Evaluate and generate report
result = trainer.predict(test_data)
actual = test_df["label"].values
predictions = np.argmax(result.predictions, axis=1)

print("\nAccuracy :", accuracy_score(actual, predictions))
print("Precision:", precision_score(actual, predictions, zero_division=0))
print("Recall   :", recall_score(actual, predictions, zero_division=0))
print("F1 Score :", f1_score(actual, predictions, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    actual, predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(actual, predictions))

# 9. Predict new customer feedback
feedback = "The product arrived damaged and I want a replacement."

inputs = tokenizer(
    feedback, return_tensors="pt",
    truncation=True, padding=True, max_length=64
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    output = model(**inputs)

prob = torch.softmax(output.logits, dim=1)
prediction = prob.argmax(dim=1).item()

classes = ["Negative", "Positive"]

print("\nPredicted Class:", classes[prediction])
print("Confidence:", round(prob[0][prediction].item() * 100, 2), "%")

# 10. Save trained model
model.save_pretrained("./bert_customer_model")
tokenizer.save_pretrained("./bert_customer_model")

print("\nModel saved successfully!")

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Started...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.615294,0.700000,0.746479,0.557895,0.638554



Accuracy : 0.7
Precision: 0.7464788732394366
Recall   : 0.5578947368421052
F1 Score : 0.6385542168674698

Classification Report:
              precision    recall  f1-score   support

    Negative       0.67      0.83      0.74       105
    Positive       0.75      0.56      0.64        95

    accuracy                           0.70       200
   macro avg       0.71      0.69      0.69       200
weighted avg       0.71      0.70      0.69       200

Confusion Matrix:
[[87 18]
 [42 53]]

Predicted Class: Negative
Confidence: 65.85 %


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
